# Accounts 下载前的 Companies 样本筛选

目的：在不上网、不下载 accounts 文件之前，先用现有 `UKcompanies_8_sectors_cleaned.csv` 判断哪些公司最值得优先下载 Companies House accounts 文件。

这个 notebook 会输出到当前 Git 仓库内的 `00 Data Preparation + EDA/Financial Data/Small Scale Data Test/Sample Companies`。

输出文件包括：

- `financial_screening_summary.csv`：总体覆盖率/筛选概览
- `account_category_summary.csv`：不同 account category 的分布与候选数量
- `sector_category_summary.csv`：行业 x account category 分布
- `accounts_download_candidates_top.csv`：优先下载 accounts 的候选公司
- `accounts_download_sample_stratified.csv`：按行业和 account category 分层抽样样本

注意：这里不能验证 turnover 是否真实存在，只能判断哪些公司更可能有可解析 accounts 文件。真实 turnover 需要下载并解析 iXBRL/XBRL 文件。


In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np


def find_project_root():
    """Find the cloned Git repository root from the current notebook location."""
    candidates = []
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    candidates.append(Path.cwd())

    for start in candidates:
        start = start.resolve()
        for path in [start, *start.parents]:
            if (path / ".git").exists() and (path / "00 Data Preparation + EDA").exists():
                return path

    raise FileNotFoundError(
        "Cannot locate project root. Run this notebook inside the cloned repository, "
        "or set environment variable PROJECT_ROOT to the repo root."
    )


def first_existing(paths, label):
    """Return the first existing path, with a clear error if none are found."""
    paths = [Path(path) for path in paths]
    for path in paths:
        if path.exists():
            return path
    tried = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


PROJECT_ROOT = find_project_root()
FINANCIAL_DIR = PROJECT_ROOT / "00 Data Preparation + EDA" / "Financial Data"
SMALL_TEST_DIR = FINANCIAL_DIR / "Small Scale Data Test"
SAMPLE_COMPANIES_DIR = SMALL_TEST_DIR / "Sample Companies"
LOCAL_DATA_ROOT = Path(os.environ.get(
    "LOCAL_DATA_ROOT",
    r"E:\000硕士毕设\财务数据\Local Large Data",
))
ACCOUNTS_ZIP_DIR = first_existing(
    [
        LOCAL_DATA_ROOT / "Accounts Data_2025.1_2026.5",
        FINANCIAL_DIR / "Accounts Data_2025.1_2026.5",
    ],
    "accounts ZIP directory",
)

INPUT_CSV = first_existing(
    [
        LOCAL_DATA_ROOT / "UKcompanies_8_sectors_cleaned.csv",
        SAMPLE_COMPANIES_DIR / "UKcompanies_8_sectors_cleaned.csv",
        FINANCIAL_DIR / "UKcompanies_8_sectors_cleaned.csv",
        PROJECT_ROOT / "UKcompanies_8_sectors_cleaned.csv",
    ],
    "UKcompanies_8_sectors_cleaned.csv",
)
OUTPUT_DIR = SAMPLE_COMPANIES_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 200_000
RANDOM_STATE = 42
MAX_TOP_CANDIDATES = 50_000
TOP_KEEP_PER_CHUNK = 8_000
SAMPLE_PER_GROUP = 300
RECENT_ACCOUNTS_CUTOFF = pd.Timestamp("2024-01-01")

print("Project root:", PROJECT_ROOT)
print("Local data root:", LOCAL_DATA_ROOT)
print("Input:", INPUT_CSV)
print("Output:", OUTPUT_DIR)


In [ ]:
USECOLS = [
    "CompanyNumber",
    "CompanyName",
    "CompanyStatus",
    "CompanyCategory",
    "CountryOfOrigin",
    "RegAddress_Country",
    "RegAddress_PostTown",
    "RegAddress_PostCode",
    "IncorporationDate",
    "primary_sic_code",
    "matched_sectors",
    "has_fast_growth",
    "primary_sector",
    "sector_id",
    "Accounts_AccountCategory",
    "Accounts_LastMadeUpDate",
    "Accounts_NextDueDate",
    "Mortgages_NumMortOutstanding",
    "Mortgages_NumMortCharges",
    "has_outstanding_charges",
    "company_age_years",
    "CountryOfOrigin_clean",
    "is_uk_company",
]

HIGH_ACCOUNT_CATEGORIES = {"FULL", "GROUP", "MEDIUM"}

MEDIUM_ACCOUNT_CATEGORIES = {
    "SMALL",
    "TOTAL EXEMPTION FULL",
    "TOTAL EXEMPTION SMALL",
    "UNAUDITED ABRIDGED",
    "AUDITED ABRIDGED",
    "AUDIT EXEMPTION SUBSIDIARY",
    "FILING EXEMPTION SUBSIDIARY",
    "PARTIAL EXEMPTION",
}

LOW_ACCOUNT_CATEGORIES = {"MICRO ENTITY"}
EXCLUDE_ACCOUNT_CATEGORIES = {"DORMANT", "NO ACCOUNTS FILED", "ACCOUNTS TYPE NOT AVAILABLE"}

NON_MAINLAND_POSTCODE_PREFIXES = ("BT", "JE", "GY", "IM")


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().map({
        "true": True,
        "1": True,
        "yes": True,
        "false": False,
        "0": False,
        "no": False,
    }).fillna(False)


def classify_account_category(cat: pd.Series) -> pd.Series:
    cat_clean = cat.fillna("").astype("string").str.strip().str.upper()
    return np.select(
        [
            cat_clean.isin(HIGH_ACCOUNT_CATEGORIES),
            cat_clean.isin(MEDIUM_ACCOUNT_CATEGORIES),
            cat_clean.isin(LOW_ACCOUNT_CATEGORIES),
            cat_clean.isin(EXCLUDE_ACCOUNT_CATEGORIES),
        ],
        ["high_likelihood", "medium_likelihood", "low_likelihood", "exclude_or_no_accounts"],
        default="unknown",
    )


def classify_download_priority(df: pd.DataFrame) -> pd.Series:
    high = (
        df["is_active"]
        & df["is_mainland_uk_approx"]
        & df["has_accounts_last_made_up"]
        & df["is_recent_accounts"]
        & df["account_category_group"].isin(["high_likelihood", "medium_likelihood"])
    )
    medium = (
        df["is_active"]
        & df["is_mainland_uk_approx"]
        & df["has_accounts_last_made_up"]
        & df["account_category_group"].eq("low_likelihood")
    )
    coverage_only = (
        df["is_active"]
        & df["is_mainland_uk_approx"]
        & df["account_category_group"].isin(["exclude_or_no_accounts", "unknown"])
    )
    return np.select(
        [high, medium, coverage_only],
        ["priority_download", "coverage_sample", "low_priority_check"],
        default="exclude_from_initial_download",
    )


def infer_segment_proxy(df: pd.DataFrame) -> pd.DataFrame:
    inferred = np.select(
        [
            df["Accounts_AccountCategory_clean"].isin(["MICRO ENTITY"]),
            df["Accounts_AccountCategory_clean"].isin(["DORMANT"]),
            df["Accounts_AccountCategory_clean"].isin(["NO ACCOUNTS FILED"]),
            df["Accounts_AccountCategory_clean"].isin(["FULL", "GROUP", "MEDIUM"]),
            df["Accounts_AccountCategory_clean"].isin(["SMALL", "TOTAL EXEMPTION FULL", "UNAUDITED ABRIDGED"]),
        ],
        ["likely_BB", "out_of_scope_or_dormant", "unknown_new_or_missing", "possible_SME_or_Mid", "likely_BB_or_SME"],
        default="unknown",
    )
    confidence = np.select(
        [
            df["Accounts_AccountCategory_clean"].isin(["MICRO ENTITY"]),
            df["Accounts_AccountCategory_clean"].isin(["FULL", "GROUP", "MEDIUM"]),
            df["Accounts_AccountCategory_clean"].isin(["NO ACCOUNTS FILED", "DORMANT"]),
        ],
        ["medium", "medium", "low"],
        default="low",
    )
    df["inferred_lloyds_segment_proxy"] = inferred
    df["segment_confidence"] = confidence
    return df


def add_screening_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["CompanyNumber"] = df["CompanyNumber"].astype("string").str.zfill(8)
    df["Accounts_AccountCategory_clean"] = df["Accounts_AccountCategory"].fillna("").astype("string").str.strip().str.upper()
    df["CompanyStatus_clean"] = df["CompanyStatus"].fillna("").astype("string").str.strip()
    df["RegAddress_PostCode_clean"] = df["RegAddress_PostCode"].fillna("").astype("string").str.upper().str.strip()

    df["is_uk_company_bool"] = to_bool(df["is_uk_company"])
    df["has_fast_growth_bool"] = to_bool(df["has_fast_growth"])
    df["has_outstanding_charges_bool"] = to_bool(df["has_outstanding_charges"])

    df["accounts_last_made_up_date"] = pd.to_datetime(df["Accounts_LastMadeUpDate"], errors="coerce")
    df["accounts_next_due_date"] = pd.to_datetime(df["Accounts_NextDueDate"], errors="coerce")
    df["incorporation_date"] = pd.to_datetime(df["IncorporationDate"], errors="coerce")

    df["account_category_group"] = classify_account_category(df["Accounts_AccountCategory_clean"])
    df["is_active"] = df["CompanyStatus_clean"].eq("Active")
    df["has_accounts_last_made_up"] = df["accounts_last_made_up_date"].notna()
    df["is_recent_accounts"] = df["accounts_last_made_up_date"].ge(RECENT_ACCOUNTS_CUTOFF)

    postcode_prefix = df["RegAddress_PostCode_clean"].str.extract(r"^([A-Z]{1,2})", expand=False).fillna("")
    df["is_mainland_uk_approx"] = df["is_uk_company_bool"] & ~postcode_prefix.str.startswith(NON_MAINLAND_POSTCODE_PREFIXES)

    outstanding = pd.to_numeric(df["Mortgages_NumMortOutstanding"], errors="coerce").fillna(0)
    charges = pd.to_numeric(df["Mortgages_NumMortCharges"], errors="coerce").fillna(0)
    age = pd.to_numeric(df["company_age_years"], errors="coerce").fillna(0)

    base_score = np.select(
        [
            df["account_category_group"].eq("high_likelihood"),
            df["account_category_group"].eq("medium_likelihood"),
            df["account_category_group"].eq("low_likelihood"),
            df["account_category_group"].eq("exclude_or_no_accounts"),
        ],
        [60, 45, 25, 0],
        default=10,
    )

    df["accounts_download_score"] = (
        base_score
        + np.where(df["is_active"], 15, -20)
        + np.where(df["is_mainland_uk_approx"], 10, -15)
        + np.where(df["has_accounts_last_made_up"], 10, -10)
        + np.where(df["is_recent_accounts"], 10, 0)
        + np.where(df["has_outstanding_charges_bool"], 5, 0)
        + np.where(charges.gt(0), np.minimum(charges, 10), 0)
        + np.where(outstanding.gt(0), 3, 0)
        + np.where(df["has_fast_growth_bool"], 5, 0)
        + np.where(age.ge(2), 2, 0)
    )

    df["download_priority"] = classify_download_priority(df)
    df = infer_segment_proxy(df)

    df["screening_reason"] = (
        "category=" + df["Accounts_AccountCategory_clean"].fillna("")
        + "; priority=" + df["download_priority"].fillna("")
        + "; recent_accounts=" + df["is_recent_accounts"].astype(str)
        + "; active=" + df["is_active"].astype(str)
        + "; mainland_uk_approx=" + df["is_mainland_uk_approx"].astype(str)
    )

    return df


OUTPUT_COLUMNS = [
    "CompanyNumber",
    "CompanyName",
    "CompanyStatus",
    "CompanyCategory",
    "RegAddress_PostTown",
    "RegAddress_PostCode",
    "IncorporationDate",
    "primary_sic_code",
    "primary_sector",
    "Accounts_AccountCategory",
    "Accounts_LastMadeUpDate",
    "Accounts_NextDueDate",
    "Mortgages_NumMortOutstanding",
    "Mortgages_NumMortCharges",
    "has_outstanding_charges",
    "company_age_years",
    "account_category_group",
    "download_priority",
    "accounts_download_score",
    "inferred_lloyds_segment_proxy",
    "segment_confidence",
    "screening_reason",
]

## 执行筛选

这一步会分块读取大 CSV，不会把 341 万行一次性全部塞进内存。


In [ ]:
overall_counts = []
category_summaries = []
sector_category_summaries = []
top_candidate_chunks = []
sample_candidate_chunks = []

reader = pd.read_csv(
    INPUT_CSV,
    usecols=USECOLS,
    dtype="string",
    chunksize=CHUNK_SIZE,
    low_memory=False,
)

for chunk_idx, chunk in enumerate(reader, start=1):
    df = add_screening_features(chunk)

    overall_counts.append({
        "chunk": chunk_idx,
        "rows": len(df),
        "active_rows": int(df["is_active"].sum()),
        "mainland_uk_approx_rows": int(df["is_mainland_uk_approx"].sum()),
        "has_accounts_last_made_up_rows": int(df["has_accounts_last_made_up"].sum()),
        "recent_accounts_rows": int(df["is_recent_accounts"].sum()),
        "priority_download_rows": int(df["download_priority"].eq("priority_download").sum()),
        "coverage_sample_rows": int(df["download_priority"].eq("coverage_sample").sum()),
    })

    cat_summary = (
        df.groupby(["Accounts_AccountCategory_clean", "account_category_group", "download_priority"], dropna=False)
        .size()
        .reset_index(name="rows")
    )
    category_summaries.append(cat_summary)

    sector_cat_summary = (
        df.groupby(["primary_sector", "Accounts_AccountCategory_clean", "account_category_group", "download_priority"], dropna=False)
        .size()
        .reset_index(name="rows")
    )
    sector_category_summaries.append(sector_cat_summary)

    priority = df[df["download_priority"].eq("priority_download")]
    if not priority.empty:
        top_candidate_chunks.append(
            priority.nlargest(min(TOP_KEEP_PER_CHUNK, len(priority)), "accounts_download_score")[OUTPUT_COLUMNS]
        )

    sample_base = df[df["download_priority"].isin(["priority_download", "coverage_sample"])]
    if not sample_base.empty:
        chunk_sample = (
            sample_base
            .groupby(["primary_sector", "account_category_group"], group_keys=False, dropna=False)
            .apply(lambda g: g.sample(n=min(len(g), SAMPLE_PER_GROUP), random_state=RANDOM_STATE + chunk_idx))
        )
        sample_candidate_chunks.append(chunk_sample[OUTPUT_COLUMNS])

    if chunk_idx % 5 == 0:
        print(f"Processed {chunk_idx} chunks")

print("Done")

In [ ]:
overall = pd.DataFrame(overall_counts)
summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "active_rows",
        "mainland_uk_approx_rows",
        "has_accounts_last_made_up_rows",
        "recent_accounts_rows",
        "priority_download_rows",
        "coverage_sample_rows",
    ],
    "value": [
        int(overall["rows"].sum()),
        int(overall["active_rows"].sum()),
        int(overall["mainland_uk_approx_rows"].sum()),
        int(overall["has_accounts_last_made_up_rows"].sum()),
        int(overall["recent_accounts_rows"].sum()),
        int(overall["priority_download_rows"].sum()),
        int(overall["coverage_sample_rows"].sum()),
    ],
})
summary["share_of_total"] = summary["value"] / summary.loc[summary["metric"].eq("total_rows"), "value"].iloc[0]

category_summary = (
    pd.concat(category_summaries, ignore_index=True)
    .groupby(["Accounts_AccountCategory_clean", "account_category_group", "download_priority"], dropna=False)["rows"]
    .sum()
    .reset_index()
    .sort_values("rows", ascending=False)
)

sector_category_summary = (
    pd.concat(sector_category_summaries, ignore_index=True)
    .groupby(["primary_sector", "Accounts_AccountCategory_clean", "account_category_group", "download_priority"], dropna=False)["rows"]
    .sum()
    .reset_index()
    .sort_values(["primary_sector", "rows"], ascending=[True, False])
)

if top_candidate_chunks:
    top_candidates = (
        pd.concat(top_candidate_chunks, ignore_index=True)
        .drop_duplicates(subset=["CompanyNumber"], keep="first")
        .nlargest(MAX_TOP_CANDIDATES, "accounts_download_score")
        .reset_index(drop=True)
    )
else:
    top_candidates = pd.DataFrame(columns=OUTPUT_COLUMNS)

if sample_candidate_chunks:
    sample_pool = pd.concat(sample_candidate_chunks, ignore_index=True).drop_duplicates(subset=["CompanyNumber"], keep="first")
    stratified_sample = (
        sample_pool
        .groupby(["primary_sector", "account_category_group"], group_keys=False, dropna=False)
        .apply(lambda g: g.sample(n=min(len(g), SAMPLE_PER_GROUP), random_state=RANDOM_STATE))
        .sort_values(["primary_sector", "account_category_group", "accounts_download_score"], ascending=[True, True, False])
        .reset_index(drop=True)
    )
else:
    stratified_sample = pd.DataFrame(columns=OUTPUT_COLUMNS)

summary.to_csv(OUTPUT_DIR / "financial_screening_summary.csv", index=False, encoding="utf-8-sig")
category_summary.to_csv(OUTPUT_DIR / "account_category_summary.csv", index=False, encoding="utf-8-sig")
sector_category_summary.to_csv(OUTPUT_DIR / "sector_category_summary.csv", index=False, encoding="utf-8-sig")
top_candidates.to_csv(OUTPUT_DIR / "accounts_download_candidates_top.csv", index=False, encoding="utf-8-sig")
stratified_sample.to_csv(OUTPUT_DIR / "accounts_download_sample_stratified.csv", index=False, encoding="utf-8-sig")

print("Saved files to", OUTPUT_DIR)
print("Top candidates:", len(top_candidates))
print("Stratified sample:", len(stratified_sample))

In [ ]:
display(summary)
display(category_summary.head(20))
display(top_candidates.head(20))

## 输出结果怎么用

推荐下一步：

1. 先看 `financial_screening_summary.csv`，确认有多少公司进入 `priority_download`。
2. 看 `account_category_summary.csv`，确认 FULL/GROUP/MEDIUM/SMALL 等类别的数量。
3. 用 `accounts_download_sample_stratified.csv` 做小样本下载和解析试验。
4. 等 parser 跑通后，再用 `accounts_download_candidates_top.csv` 做批量 accounts ZIP manifest 匹配。

注意：`inferred_lloyds_segment_proxy` 只是基于 account category 的 proxy，不是 Lloyds 的真实 turnover segment。真实 BB/SME/Mid Corporate 分层需要解析 accounts 文件里的 turnover。
